# wandb-log-step composite — cx18: wandb.log then zero_grad(set_to_none=True) — standard order

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `wandb-log-step`, `zero-grad-set-none`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F
import wandb

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "wandb-log-step"
DD_ATOM_IDS = ["wandb-log-step", "zero-grad-set-none"]
DD_SUBTOPICS = ["Logging: wandb.log step", "PyTorch: zero_grad"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

After `optimizer.step()`, two cleanup actions remain: (a) emit the metrics for this batch to wandb, (b) wipe the gradients for the next batch. The canonical order is **log first, then zero_grad**:

```python
loss.backward()
optimizer.step()
wandb.log({'loss': loss.item()}, step=step_count)  # atom A.
optimizer.zero_grad(set_to_none=True)              # atom B.
```

**Atom A — `wandb-log-step`.** `wandb.log({...}, step=N)` plots metrics against `N`. Pass `loss.item()` (Python float), `step` as `int`.

**Atom B — `zero-grad-set-none`.** Sets every `p.grad = None` so the next backward allocates fresh tensors with no accumulation.

**Why log first, then zero_grad.** The log line may want to record gradient-norm statistics in the future (`wandb.log({'loss': l, 'grad_norm': gn}, step=N)`). If you `zero_grad` first, every grad is `None` by the time you'd compute the norm — you've destroyed the data you wanted to log. Even if today's metric is just `loss`, the order leaves the door open for richer logging without a refactor.

**A weaker reason:** consistency. The order `loss → backward → step → log → zero_grad` matches what every ARENA trainer template does, so reviewers can pattern-match.

We test by mocking wandb and asserting (i) `wandb.log` is called with the right metrics+step, (ii) AFTER the function returns every `p.grad is None`, and (iii) the call ORDER is correct — gradient norm is logged with a non-zero value, which only works if log fires BEFORE zero_grad.

### Composite Exercise — wandb.log then zero_grad(set_to_none=True) — standard order

**Atoms exercised together**: `wandb-log-step`, `zero-grad-set-none`

Implement `cx18_train_step_with_log(model, optimizer, x, y, loss_fn, step_count)`.

Sequence (one batch):
1. `pred = model(x)`; `loss = loss_fn(pred, y)`.
2. `loss.backward()`.
3. `optimizer.step()`.
4. Compute `grad_norm`: the L2 norm across ALL parameter gradients flattened together — `torch.cat([p.grad.flatten() for p in model.parameters()]).norm().item()`. (This relies on `.grad` still being a Tensor — the whole point of doing this BEFORE `zero_grad`.)
5. Call `wandb.log({'loss': loss.item(), 'grad_norm': grad_norm}, step=step_count)` (atom A).
6. Call `optimizer.zero_grad(set_to_none=True)` (atom B).
7. Return `loss.item()`.

**Test asserts** (via mocked wandb):
- `wandb.log` called exactly once with both `loss` and `grad_norm` keys.
- `grad_norm` value is a positive float (proof you computed it BEFORE zero_grad).
- Step kwarg equals `step_count`.
- After return, every `p.grad is None`.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx18_train_step_with_log(model, optimizer, x, y, loss_fn, step_count):
    """One step: forward → backward → step → log → zero_grad. Returns loss float."""
    raise NotImplementedError

def _test_cx18():
    # Case A: round-trip with grad_norm — must be computed BEFORE zero_grad.
    wandb.log.reset_mock()
    t.manual_seed(0)
    model = nn.Linear(3, 1)
    opt = t.optim.SGD(model.parameters(), lr=0.05)
    loss_fn = nn.MSELoss()
    x = t.randn(8, 3)
    y = t.randn(8, 1)

    loss_val = cx18_train_step_with_log(model, opt, x, y, loss_fn, step_count=42)

    assert isinstance(loss_val, float)
    assert wandb.log.call_count == 1, f'expected 1 wandb.log call, got {wandb.log.call_count}'
    call = wandb.log.call_args_list[0]
    metrics = call.args[0] if call.args else call.kwargs.get('data')
    assert isinstance(metrics, dict)
    assert 'loss' in metrics and 'grad_norm' in metrics, (
        f'metrics dict missing keys; got {list(metrics.keys())}'
    )
    assert isinstance(metrics['loss'], float)
    assert isinstance(metrics['grad_norm'], float)
    assert metrics['grad_norm'] > 0, (
        f'grad_norm should be > 0 (computed BEFORE zero_grad); got {metrics["grad_norm"]}. '
        f'If you got 0 or got an AttributeError, you called zero_grad BEFORE computing grad_norm.'
    )
    assert call.kwargs.get('step') == 42, f"step kwarg expected 42, got {call.kwargs.get('step')}"

    # Case B: after return, all grads are None (set_to_none observed externally).
    for p in model.parameters():
        assert p.grad is None, f'post-return p.grad must be None; got {p.grad!r}'

    # Case C: chained calls — log fires each call, grad_norm always > 0.
    wandb.log.reset_mock()
    t.manual_seed(1)
    model2 = nn.Linear(3, 1)
    opt2 = t.optim.SGD(model2.parameters(), lr=0.05)
    for step_n in [1, 2, 3]:
        cx18_train_step_with_log(model2, opt2, t.randn(8, 3), t.randn(8, 1), loss_fn, step_count=step_n)
    assert wandb.log.call_count == 3
    for i, call in enumerate(wandb.log.call_args_list):
        metrics_i = call.args[0] if call.args else call.kwargs.get('data')
        assert metrics_i['grad_norm'] > 0, (
            f'call {i}: grad_norm must be > 0 every step (computed before zero_grad)'
        )
        assert call.kwargs.get('step') == i + 1

    # Case D: order matters — if log were AFTER zero_grad, p.grad would be None during the
    # norm computation, raising AttributeError. The fact that Case A passed without exception
    # AND grad_norm > 0 is the joint signal that the order is right.
    _dd_passed.add('cx18')

_test_cx18()

<details><summary>Show solution — cx18</summary>

```python
def cx18_train_step_with_log(model, optimizer, x, y, loss_fn, step_count):
    pred = model(x)
    loss = loss_fn(pred, y)
    loss.backward()
    optimizer.step()
    # grad_norm BEFORE zero_grad — relies on .grad still being a Tensor.
    grad_norm = t.cat([p.grad.flatten() for p in model.parameters()]).norm().item()
    # Atom A: wandb.log with both metrics + step kwarg.
    wandb.log({'loss': loss.item(), 'grad_norm': grad_norm}, step=step_count)
    # Atom B: zero-grad-set-none.
    optimizer.zero_grad(set_to_none=True)
    return loss.item()
```

The grad-norm metric is the canonical reason for log-before-zero_grad — it's a common diagnostic in GAN/VAE trainers where one of the two networks may be exploding while the other looks fine. ARENA's reference DCGAN trainer uses exactly this pattern. If you tried `zero_grad → log` and Case A raised `AttributeError: NoneType has no attribute 'flatten'`, that's the canonical bug — the order in the canonical recipe is there for a reason.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx18'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx18',
        'subtopics': ["Logging: wandb.log step", "PyTorch: zero_grad"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()